# 103. Integral Neural Networks from the authors' code, and S-Net HY beside it

Two methods, one per card. GPU 0: Integral Neural Networks (Solodskikh
et al., CVPR 2023) run from the authors' repository (TorchIntegral). GPU 1:
S-Net HY, the third seed of S-Net on MobileNetV2 slim_last, from our
train.py (its other seeds are in notebook 106); it is here so that no
card stays idle.

INN turns a trained ordinary network into a continuous one and fine-tunes
it while sampling sizes; any width is then read by resampling. Two
phases, one after the other on GPU 0, seed 1995:

| | GPU | phase | code |
|---|---|---|---|
| inn_base | 0 | a plain CIFAR ResNet-50 (ordinary torch.nn, the channel counts of our US-Net rows at width 1.0) on our recipe: 100 epochs, batch 256, SGD nesterov lr 0.2, 5 warm-up epochs then cosine, wd 5e-4, clip 5. Prints its width-1.00 line: the individual network at full width | TheStageAI/TorchIntegral at 84c5e89 + `third_party/patches/inn.patch` |
| inn_r50 | 0 | their `IntegralWrapper` on that checkpoint (TSP channel permutation, continuous weights, BN fused as in their default), then their fine-tuning recipe: Adam 1e-4, batch 128, 50 epochs, x0.33 at 10/20/30/40. One width w ~ U(0.25, 1) per step for every group, then the sixteen widths by resampling | same |

Not on the same footing as the 100-epoch rows: 150 epochs in all (100 SGD +
50 Adam), and no BatchNorm at any width (fused), so no recalibration. The
authors never resampled a whole network down to 25 %; converted but not
yet fine-tuned, every width under 1.00 is near chance, so the fine-tuning
has to teach the narrow widths from scratch.
`third_party/patches/inn_README.md` lists every change and deviation.

| | GPU | method | code |
|---|---|---|---|
| hy_snet_mbv2_slimlast_seed3 | 1 | Slimmable Networks (ICLR 2019) on our CIFAR MobileNetV2 with slim_last, seed 2006: four switches 0.25/0.5/0.75/1.00 on every batch, a private BN per switch, cross entropy only, one shared classifier; our recipe; read at its four switches from the last epoch, no calibration | our train.py, `apps/cifar100_hy_snet_mbv2_slimlast_seed3.yml`; a two-epoch smoke runs first |

## More than one session

INN: estimated about 5 h for the base and 3.5 h for the fine-tuning on a
T4 (not measured there), plus 5-10 minutes for the permutation search.
HY: about a US-Net run of its setting. Close to one session. INN writes
`ckpt/<label>/last.pt` and HY `new-pruning/logs/cifar100_<branch>/latest_checkpoint.pt`
after every epoch. When the session ends, Save Version, then in a new run
attach that version's output (pick the version by number, not Latest) and
run again: the notebook finds the checkpoints, a finished base is not
trained again, and both runs continue. Attach the CIFAR-100 dataset as
well.

In [ ]:
# Fixed for this notebook.
REPO_URL = 'https://github.com/duyh80456-code/new-pruning.git'
REPO_BRANCH = 'nhan'
# key: (repository, pinned commit, folder, patch, extra pip packages)
UPSTREAMS = {'inn': ('https://github.com/TheStageAI/TorchIntegral.git', '84c5e89047f1d47bbf6c62065aa223d997f9b5e7', 'TorchIntegral', 'inn.patch', 'cython')}
WORKER_ENV = {'INN_WORKERS': '2'}

In [ ]:
import os, queue, re, shutil, subprocess, sys, threading, time
import torch

n_gpu = torch.cuda.device_count()
print('torch', torch.__version__, '| gpus', n_gpu)
if n_gpu < 2:
    raise SystemExit('Needs GPU T4 x2: one card per method.')

WORK = '/kaggle/working'
OURS = os.path.join(WORK, 'new-pruning')
if not os.path.isdir(OURS):
    subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, OURS],
                   check=True)
print('our code at', subprocess.run(
    ['git', '-C', OURS, 'log', '-1', '--format=%h %s'],
    capture_output=True, text=True).stdout.strip())

CODE = {}
packages = []
for key, (url, commit, folder, patch, pip) in UPSTREAMS.items():
    CODE[key] = os.path.join(WORK, folder)
    if not os.path.isdir(CODE[key]):
        subprocess.run(['git', 'clone', url, CODE[key]], check=True)
        subprocess.run(['git', '-C', CODE[key], 'checkout', '-q', commit],
                       check=True)
        subprocess.run(['git', '-C', CODE[key], 'apply', os.path.join(
            OURS, 'third_party', 'patches', patch)], check=True)
    print(key, 'at', subprocess.run(
        ['git', '-C', CODE[key], 'log', '-1', '--format=%h %ad',
         '--date=short'], capture_output=True, text=True).stdout.strip(),
        '+', patch)
    packages += pip.split()

# install without letting pip replace Kaggle's torch
base = torch.__version__.split('+')[0]
with open(os.path.join(WORK, 'constraints.txt'), 'w') as handle:
    handle.write('torch=={}\n'.format(base))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c',
                os.path.join(WORK, 'constraints.txt')] + packages, check=True)

In [ ]:
# the folder that contains cifar-100-python/, or a place to download it
DATA = None
for root, dirs, _ in os.walk('/kaggle/input', followlinks=True):
    if 'cifar-100-python' in dirs:
        DATA = root
        break
if DATA is None:
    DATA = os.path.join(WORK, 'data')
    os.makedirs(DATA, exist_ok=True)
    print('no CIFAR-100 attached: it will be downloaded into', DATA)
else:
    print('CIFAR-100 from', DATA)

In [ ]:
VAL_LINE = re.compile(
    r'val\s+([0-9.]+)\s+(-1|\d+)/(\d+):\s+loss:\s+([0-9.eE+-]+),\s+'
    r'top1_error:\s+([0-9.]+)')
TAG = re.compile(r'^\[([^\]\s]+)\]')
results = {}


def run_pinned(jobs):
    """one command per card, output tagged and interleaved"""
    lines = queue.Queue()
    procs = {}

    def pump(label, proc):
        for line in proc.stdout:
            lines.put((label, line.rstrip('\n')))
        proc.wait()
        lines.put((label, None))

    for label, gpu, cwd, command in jobs:
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, **WORKER_ENV)
        proc = subprocess.Popen(command, cwd=cwd, env=env, text=True,
                                stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT)
        procs[label] = proc
        threading.Thread(target=pump, args=(label, proc),
                         daemon=True).start()
        print('[{}] gpu {}: {}'.format(label, gpu, ' '.join(command)),
              flush=True)
    started, remaining = time.time(), len(jobs)
    while remaining:
        label, line = lines.get()
        if line is None:
            remaining -= 1
            print('[{}] exit code {} after {:.0f} min'.format(
                label, procs[label].returncode,
                (time.time() - started) / 60), flush=True)
            continue
        found = VAL_LINE.search(line)
        if found:
            width, epoch, total, loss, top1 = found.groups()
            if epoch == '-1' or int(epoch) == int(total) - 1:
                tag = TAG.match(line)
                results.setdefault(tag.group(1) if tag else label, {})[
                    round(float(width), 2)] = (float(loss), float(top1))
        print('[{}] {}'.format(label, line), flush=True)
    return {label: proc.returncode for label, proc in procs.items()}

In [ ]:
CK = os.path.join(WORK, 'ckpt')
# INN keeps ckpt/inn_base/last.pt and ckpt/inn_r50/last.pt
wanted = {'inn_base': ('last.pt', 'inn_base'),
          'inn_r50': ('last.pt', 'inn_r50')}
for label, (name, folder) in wanted.items():
    target = os.path.join(CK, folder or '', name)
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        for root, _, files in os.walk('/kaggle/input', followlinks=True):
            if name in files and (folder is None
                                  or os.path.basename(root) == folder):
                shutil.copy(os.path.join(root, name), target)
                print('restored', label, 'from', root)
                break
    print(label, 'resumes from its checkpoint' if os.path.exists(target)
          else 'starts at epoch 1')

In [ ]:
import shlex
# HY runs our train.py, which reads data/cifar-100-python in our repo and
# resumes from logs/cifar100_<branch>/latest_checkpoint.pt
SNET = 'hy_snet_mbv2_slimlast_seed3'
link = os.path.join(OURS, 'data', 'cifar-100-python')
if not os.path.exists(link) and os.path.isdir(
        os.path.join(DATA, 'cifar-100-python')):
    os.makedirs(os.path.dirname(link), exist_ok=True)
    os.symlink(os.path.join(DATA, 'cifar-100-python'), link)
logs = os.path.join(OURS, 'logs', 'cifar100_' + SNET)
if not os.path.exists(os.path.join(logs, 'latest_checkpoint.pt')):
    for root, _, files in os.walk('/kaggle/input', followlinks=True):
        if ('latest_checkpoint.pt' in files
                and os.path.basename(root) == 'cifar100_' + SNET):
            shutil.copytree(root, logs, dirs_exist_ok=True)
            print('restored', SNET, 'from', root)
            break
print(SNET, 'resumes from its checkpoint' if os.path.exists(
    os.path.join(logs, 'latest_checkpoint.pt')) else 'starts at epoch 1')
codes = run_pinned([(SNET + '_smoke', '1', OURS, [
    sys.executable, '-u', 'train.py', 'app:apps/smoke_{}.yml'.format(SNET)])])
if codes[SNET + '_smoke'] != 0:
    raise SystemExit('the HY smoke failed')

# GPU 0: INN phase 1, then phase 2 from its checkpoint, chained; the
# parser files each protocol line under the tag it carries
COMMON = ['--arch', 'r50', '--data_dir', DATA, '--seed', '1995']
base = ([sys.executable, '-u', 'train_base.py'] + COMMON
        + ['--ckpt_dir', os.path.join(CK, 'inn_base'),
           '--log_prefix', 'inn_base'])
inn = ([sys.executable, '-u', 'train_inn.py'] + COMMON
       + ['--base_ckpt', os.path.join(CK, 'inn_base', 'last.pt'),
          '--ckpt_dir', os.path.join(CK, 'inn_r50'),
          '--log_prefix', 'inn_r50'])
codes = run_pinned([
    ('inn_r50', '0', CODE['inn'],
     ['bash', '-c', shlex.join(base) + ' && ' + shlex.join(inn)]),
    (SNET, '1', OURS,
     [sys.executable, '-u', 'train.py',
      'app:apps/cifar100_{}.yml'.format(SNET)])])
print(codes)

## Results, against A (BX, 76.86) and K (DD, 77.50)

Send back this table and the `val ... -1/100` lines above it.

inn_base at 1.00 is the individual ResNet-50; inn_r50 is read at the
sixteen widths like every row, with the footing caveats above. HY's four
switches are read against the MobileNetV2 US-Net and SlimOT rows at the
same widths, not against the A and K columns (ResNet-50).

In [ ]:
A_REF = {0.25: 75.26, 0.3: 75.35, 0.35: 75.54, 0.4: 76.01, 0.45: 76.3, 0.5: 76.77, 0.55: 76.63, 0.6: 77.03, 0.65: 77.13, 0.7: 77.2, 0.75: 77.39, 0.8: 77.56, 0.85: 77.6, 0.9: 77.99, 0.95: 78.01, 1.0: 78.04}
K_REF = {0.25: 75.63, 0.3: 76.29, 0.35: 76.54, 0.4: 76.97, 0.45: 77.67, 0.5: 77.43, 0.55: 77.58, 0.6: 77.62, 0.65: 77.83, 0.7: 78.02, 0.75: 78.0, 0.8: 78.01, 0.85: 78.09, 0.9: 78.23, 0.95: 78.08, 1.0: 77.95}
labels = ['inn_base', 'inn_r50', 'hy_snet_mbv2_slimlast_seed3']
print('{:>7}{:>8}{:>8}'.format('width', 'A', 'K') + ''.join(
    '{:>12}{:>7}{:>7}'.format(b[:11], 'vs A', 'vs K') for b in labels))
for width in sorted(A_REF):
    row = '{:>7.2f}{:>8.2f}{:>8.2f}'.format(
        width, A_REF[width], K_REF[width])
    for label in labels:
        entry = results.get(label, {}).get(width)
        if entry is None:
            row += '{:>12}{:>7}{:>7}'.format('-', '-', '-')
            continue
        accuracy = 100.0 * (1.0 - entry[1])
        row += '{:>12.2f}{:>+7.2f}{:>+7.2f}'.format(
            accuracy, accuracy - A_REF[width], accuracy - K_REF[width])
    print(row)
reference = sum(A_REF.values()) / len(A_REF)
k_reference = sum(K_REF.values()) / len(K_REF)
print('\n{:24} mean {:.2f}'.format('A (BX)', reference))
print('{:24} mean {:.2f}'.format('K (DD)', k_reference))
for label in labels:
    table = results.get(label, {})
    if len(table) == len(A_REF):
        mean = sum(100.0 * (1.0 - v[1]) for v in table.values()) / len(table)
        print('{:24} mean {:.2f}   vs A {:+.2f}   vs K {:+.2f}'.format(
            label, mean, mean - reference, mean - k_reference))
    else:
        print('{:24} {} of 16 widths read'.format(label, len(table)))

In [ ]:
# keep the checkpoints in the output, under a name the next session finds
print('checkpoints in', CK, sorted(os.listdir(CK)))